# P0 · 3CAD with padded (not stretched) images and a cleaned training set (exploratory)

Notebook 12 found that resolution does not explain 3CAD's low scores, that most categories mix image shapes
(stretched to a square at 448 px), and that Aluminum_Camera_Cover has duplicate and leaked training images.
This run pads every image and mask to a square before resizing and drops duplicate and leaked training
images, then compares with the original 448 px run on the same categories and seeds.

**Runtime (T4):** about 4 h, mostly feature extraction. Finished runs are saved to Drive; after a disconnect,
Run all again (an interrupted category re-extracts). Needs the 3CAD archive in `MyDrive/bovw-forensics/data/`.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
import os, glob, shutil, subprocess
from bovw import ad_data

def get_mvtec():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz", "/content/mvtec"
    if not os.path.isfile(tar):
        raise SystemExit("Put mvtec_anomaly_detection.tar.xz in MyDrive/bovw-forensics/data/ (see notebook 02).")
    if not os.path.isdir(f"{local}/bottle"):
        os.makedirs(local, exist_ok=True)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return local

def get_visa():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/VisA_20220922.tar", "/content/visa"
    if not os.path.isfile(tar) or os.path.getsize(tar) < 1e9:
        ad_data.ensure_archive(tar, url=ad_data.VISA_URL, sha256=None)
    if not os.path.isdir(local):
        os.makedirs(local)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return ad_data.find_visa_root(local)

def get_3cad():
    local, archive = "/content/3cad", "/content/3cad_archive"
    if not os.path.isdir(local):
        copy = glob.glob("/content/drive/MyDrive/bovw-forensics/data/3CAD*")
        if not copy:
            raise SystemExit("Put the 3CAD archive in MyDrive/bovw-forensics/data/ (see notebook 06).")
        shutil.copy(copy[0], archive)
        os.makedirs(local)
        magic = open(archive, "rb").read(4)
        fmt = "zip" if magic[:2] == b"PK" else ("gztar" if magic[:2] == b"\x1f\x8b" else "tar")
        shutil.unpack_archive(archive, local, fmt)
        os.remove(archive)
    return ad_data.find_root(local, "Copper_Stator")

ROOT = get_3cad()
print(ROOT)

## 3 · Run

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from bovw.sweep import load_config
from bovw import anomaly

cfg = load_config("configs/p0_3cad_pad.yaml")
cfg["data"]["root"] = ROOT
pad = anomaly.run(cfg)

## 4 · Padded vs. stretched, and codebook vs. coreset when padded
Reads the saved results from Drive, so it works in a fresh session (run Setup, then this cell).

In [ ]:
import pandas as pd
from pathlib import Path
from bovw import stats
from bovw.sweep import load_results

RES = "/content/drive/MyDrive/bovw-forensics/results"
new = load_results(Path(RES) / "p0_3cad_pad.jsonl")
old = pd.read_csv(f"{RES}/p0_anomaly_3cad.csv")
old = old[old.method.isin(["codebook_dist", "coreset_knn"])]
key = ["category", "method", "k", "vocab_seed"]
for d in (new, old):
    d["k"] = d["k"].astype(int)
n_runs = new.drop_duplicates(key).groupby("category").size()
complete = n_runs[n_runs >= 18].index
print("compared:", sorted(complete), " incomplete:", sorted(set(n_runs.index) - set(complete)))
common = new[new.category.isin(complete)][key].merge(old[key]).drop_duplicates()
new_c, old_c = new.merge(common), old.merge(common)
cols = ["image_auroc", "image_auroc_top10", "pixel_auroc", "aupro"]
cmp_ = pd.concat({"stretched": old_c, "padded": new_c}, names=["resize"]).reset_index(0)
cmp_ = cmp_.groupby(["resize", "method", "k", "category"])[cols].mean().groupby(level=[0, 1, 2]).mean().unstack(0).round(3)
cmp_.to_csv(f"{RES}/p0_3cad_pad_vs_stretch.csv")
both = pd.concat([new_c.assign(method=new_c.method + "_pad"), old_c.assign(method=old_c.method + "_stretch")])
gain = pd.concat([stats.compare({"3cad": both}, f"{m}_pad", f"{m}_stretch", ks=(64, 256, 1024))
                  for m in ["codebook_dist", "coreset_knn"]])
gain.to_csv(f"{RES}/p0_3cad_pad_gain.csv", index=False)
pp = stats.compare({"3cad_pad": new_c}, "codebook_dist", "coreset_knn", ks=(64, 256, 1024))
pp.to_csv(f"{RES}/p0_3cad_pad_paired.csv", index=False)
per_cat = pd.concat({"stretched": old_c, "padded": new_c}, names=["resize"]).reset_index(0)
per_cat = per_cat[per_cat.k == 1024].pivot_table(index="category", columns=["method", "resize"], values="aupro").round(3)
per_cat.to_csv(f"{RES}/p0_3cad_pad_per_category_aupro1024.csv")
display(cmp_)
display(per_cat)
display(gain.drop(columns=["p_sign", "p_wilcoxon"]).round(3))
pp.drop(columns=["p_sign", "p_wilcoxon"]).round(3)